# `colab_runner_B.ipynb` — headless Path B runner (colab CLI, T4)

Headless twin of `runner_B.ipynb` for running via the **`colab` CLI** on a Colab VM
(not the web UI). Identical logic, with one change: the project is uploaded to
`/content` (not Google Drive), so the Drive-mount/locate cell is replaced by a
hard-coded `/content` project root. Everything else reuses the existing functions
verbatim.

Run it with: `colab exec -s <name> --timeout 5400 -f path_b/colab_runner_B.ipynb`
(after `colab upload` + untar of the `path_b/` tree and `colab install` of deps).
See `path_b/skills/COLAB_SKILL.md` for the full CLI workflow.

In [ ]:
!pip install autogluon.timeseries
!pip install -q torchvision==0.24.1 --index-url https://download.pytorch.org/whl/cu128

In [ ]:
import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

## 0. Verify AutoGluon + Chronos2 (deps installed via `colab install`)

In [ ]:
# Packages are installed out-of-band via `colab install` (uv pip) — no !pip here.
# Chronos-2 needs autogluon.timeseries >= 1.5.0.
from importlib.metadata import version
from autogluon.timeseries import TimeSeriesPredictor, TimeSeriesDataFrame
print("autogluon.timeseries", version("autogluon.timeseries"))
# Confirm the Chronos2 model key exists in this AutoGluon build (else the hyperparameters
# dict {'Chronos2': {...}} would silently no-op). If this import fails, upgrade AutoGluon.
from autogluon.timeseries.models.chronos.chronos2 import Chronos2Model
print("Chronos2Model available:", Chronos2Model.__name__)

## 0b. Locate project at `/content` (no Drive — headless CLI run)

In [ ]:
import os, sys
from pathlib import Path

# Mount Drive on Colab; locate the path_b/ dir (the dir that contains basic_cells_A.ipynb).
try:
    from google.colab import drive
    drive.mount("/content/drive")
    CANDIDATES = [
        "/content/drive/MyDrive/MOEX/Chronos2Experiment/path_b",
        "/content/drive/MyDrive/Chronos2Experiment/path_b",
        "/content/drive/MyDrive/moex-hack/Chronos2Experiment/path_b",
    ]
except Exception:
    # Local fallback: this notebook lives in /path_b/, so use its own dir / cwd.
    CANDIDATES = [str(Path.cwd()), str(Path.cwd() / "path_b")]

PATH_B_DIR = next((p for p in CANDIDATES if (Path(p) / "basic_cells_A.ipynb").exists()), None)
assert PATH_B_DIR, f"Could not find path_b/ (basic_cells_A.ipynb) in: {CANDIDATES}"
PROJECT_ROOT = str(Path(PATH_B_DIR).parent)
# chdir to PROJECT_ROOT so Path A loaders + runs/path_b/ resolve relative to the project root.
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PATH_B_DIR)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("PATH_B_DIR:", PATH_B_DIR)

## 1. Source Part A + Part B cell libraries (read-only `%run`, Part A idiom)

`%run basic_cells_A.ipynb` loads Path A's data loaders (`load_config`, `build_prefetch_manifest`,
`prefetch_all`, `load_stage_inputs`, `assemble_panels`, `resolve_cache_dir`, …) into this
namespace. `%run basic_cells_B.ipynb` then loads every Path B function (`normalize_path_b_config`,
`run_path_b`, `make_b*_hyperparameters`, …) into the *same* namespace — exactly like Part A,
no `import`.

In [ ]:
# Source Part A loaders, then Part B library (both live in path_b/; neither is modified).
%run $PATH_B_DIR/basic_cells_A.ipynb
%run $PATH_B_DIR/basic_cells_B.ipynb

# Path B functions are now in the global namespace (Part A idiom). Sanity check:
print("Path B loaded:", [n for n in ("normalize_path_b_config", "run_path_b",
      "make_b0_hyperparameters", "get_cross_learning_values") if n in dir()])

## 2. Choose a Path B config

Default is a Path-B config derived from the Path A 60m study. Switch to 10m by changing
the path. Both inherit the Path A representation and add the `path_b:` block.

In [ ]:
 CONFIG_PATH = os.path.join(PATH_B_DIR, "configs", "path_b_1d.yaml")
# CONFIG_PATH = os.path.join(PATH_B_DIR, "configs", "path_b_60m.yaml")
# CONFIG_PATH = os.path.join(PATH_B_DIR, "configs", "path_b_10m.yaml")

cfg = normalize_path_b_config(load_config(CONFIG_PATH))
print("interval:", cfg["interval"], "m | context_len:", cfg["context_len"],
      "| prediction_length:", cfg["path_b"]["prediction_length"])
print("eval_horizons:", cfg["path_b"]["eval_horizons"], "| primary:", cfg["path_b"]["primary_horizons"])
print("cross_learning arms:", get_cross_learning_values(cfg))
print("output_root:", cfg["path_b"]["output_root"], "| model_path:", cfg["path_b"]["model_path"])

## 3. Build Path A panels (reuse Path A loaders), then run Path B

Path B does **not** load ISS / Drive data itself — it inherits Path A's frozen representation.
We build `(price_panel, ret_panel, cov_panel)` exactly as Path A's `run_stage` does, then hand
them to `run_path_b` along with the AutoGluon classes.

Note: `prefetch_all` hits the MOEX ISS API on first run (rate-limited ~0.25s/req) and fills the
`/content` cache. On a CLI VM this cache is ephemeral (lost on `colab stop`).

In [ ]:
# --- Build the panels via Path A loaders (from the %run'd basic_cells) ---
cache_dir = resolve_cache_dir(cfg)                 # Path A helper
manifest = build_prefetch_manifest([cfg])          # Path A helper
prefetch_all(manifest, cache_dir)                  # idempotent; fills cache
prices, indexes, futures = load_stage_inputs(cfg, cache_dir)   # Path A cache-only load
price_panel, ret_panel, cov_panel = assemble_panels(cfg, prices, indexes, futures)  # Path A
print(f"panels: price={price_panel.shape} ret={ret_panel.shape} cov={cov_panel.shape}")
panels = (price_panel, ret_panel, cov_panel)

In [ ]:
# --- Run Path B. Choose which steps to run here (this is the run-time selector). ---
#   ("b0",)                          -> B0-only smoke test (fast; proves the pipeline)
#   ("b0", "b1")                     -> B0 + first LoRA candidate
#   ("b0", "b1", "b2", "b3", "b4")   -> full run
# Keep "b0" in the set: every B1-B4 candidate is scored as delta-vs-B0, so B0 is the anchor.
PATH_B_STEPS = ("b0", "b1", "b2", "b3", "b4")

ag_modules = {"TimeSeriesPredictor": TimeSeriesPredictor, "TimeSeriesDataFrame": TimeSeriesDataFrame}

summary = run_path_b(CONFIG_PATH, steps=PATH_B_STEPS, panels=panels, ag_modules=ag_modules)
print("\nran steps:", PATH_B_STEPS)
print("runs csv:", summary["runs_csv"])
print("comparison plots:", summary["comparison_plots_dir"])

## 4. Print the comparison table (all candidates vs B0) — headless

In [ ]:
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
comp = summary["comparison"]
cols = ["step_id", "run_id", "cross_learning", "fine_tune_steps", "fine_tune_lr",
        "batch_size", "primary_score", "primary_score_b0", "delta_vs_b0",
        "delta_h1_vs_b0", "delta_h2_vs_b0", "delta_h3_vs_b0",
        "coverage", "delta_coverage_vs_b0", "fit_seconds"]
cols = [c for c in cols if c in comp.columns]
print(comp[cols].sort_values(["step_id", "run_id"]).reset_index(drop=True).to_string())

## 5. List output artifacts (plots are downloaded, not displayed) — headless

In [ ]:
import glob

# Comparison plots (paths only; download the runs/ tarball to view them locally).
print("=== Comparison plots ===")
for p in sorted(glob.glob(os.path.join(summary["comparison_plots_dir"], "*.png"))):
    print(p)

# Per-candidate plots for one representative run (e.g. B1 main arm)
out_root = cfg["path_b"]["output_root"]
interval = cfg["interval"]
b1_plots = os.path.join(out_root, f"{interval}m", "b1_lora_min", "seed_main_cross_True", "plots")
print("\n=== B1 (main arm) per-candidate plots ===")
for p in sorted(glob.glob(os.path.join(b1_plots, "*.png"))):
    print(p)

## 6. Notes
- This is the headless CLI twin of `runner_B.ipynb`; the only logic change vs. that notebook
  is locating the project at `/content` instead of mounting Google Drive.
- To enable the diagnostic `cross_learning=False` arm: set `path_b.cross_learning_diagnostics: true`
  in the config and re-run — no code edits (every B0–B4 run then executes for both arms).
- All runs are written immutably under `runs/path_b/{interval}m/{step_id}/{run_id}/`; the
  append-only `runs/path_b/path_b_runs.csv` carries the delta-vs-B0 columns.
- Horizons are `[1, 2, 3]` and `prediction_length=3` for Path B (Path A uses `[2,3,5]`/H=5);
  every comparison is against **B0** (a re-run zero-shot at the same horizons).